In [0]:
# REQUIREMENT:
    
# For each employee, look at their monthlastdate values in chronological order. Group consecutive month-end dates into unbroken "streaks" (i.e., no calendar month is skipped between one row and the next). Once a gap is found (a month is missing), the streak ends. Return the last record of the first consecutive streak for each employee — including its commissionamt and monthlastdate.

# This is essentially a classic "gaps and islands" problem: identify contiguous groups ("islands") of monthly records per employee, and return the last row of the first island

print("-----GIVEN DATAFRAME-----\n\n\n")

data = [
    (1, 300, "31-Jan-2021"),
    (1, 400, "28-Feb-2021"),
    (1, 200, "31-Mar-2021"),
    (2, 1000, "31-Oct-2021"),
    (2, 900, "31-Dec-2021")
]

# Create df
df = spark.createDataFrame(
    data,
    ["empid", "commissionamt", "monthlastdate"]
)

df.show()


print("-----REQUIRED DATAFRAME-----\n\n\n")

data1 = [
    (1, 200, "31-Mar-2021"),
    (2, 1000, "31-Oct-2021")
]

# Create df1
df1 = spark.createDataFrame(
    data1,
    ["empid", "commissionamt", "monthlastdate"]
)

df1.show()


print("-----SOLUTION-----\n\n\n")

##Extract month from date

from pyspark.sql.functions import *
from pyspark.sql.window import Window

##Converting date from string to date type

datedf = df.withColumn("date", to_date(df.monthlastdate, "dd-MMM-yyyy"))
datedf.show()

##Partitioning by empid ordered by date

w = Window.partitionBy("empid").orderBy("date")
rowdf = datedf.withColumn("rn", row_number().over(w))
rowdf.show()
rowdf.printSchema()

##Subtracting the month by row number yields a uniform date for a streak and it breaks when a streak ends

diffdf = rowdf.withColumn("diffdate", add_months(trunc("date","MM") , -col("rn"))) 
diffdf.show()

##The first value in each group

w2 = Window.partitionBy("empid").orderBy("date")
findf = diffdf.withColumn("firststreak", first("diffdate").over(w2))
findf.show()

##Filter

fildf = findf.select("empid","commissionamt","monthlastdate","date").filter(col("diffdate")==col("firststreak"))
fildf.show()

##Selecting the max date for streak ending

w3 = Window.partitionBy("empid").orderBy(col("date").desc())

soldf = fildf.withColumn("rownum", row_number().over(w3))\
    .filter(col("rownum")==1)\
        .drop("date","rownum")

soldf.show()

##Logic

# For each employee, the records are ordered chronologically and assigned a row number. The key idea is that if you subtract the row number (in months) from the actual month of each record, consecutive months will always cancel out to the same reference value — because both the date and the row number advance by exactly one in lockstep. The moment a month is skipped, the date moves ahead faster than the row number, causing this reference value to shift. That shift is what marks the boundary between one streak and the next.

# Using this, each row gets tagged with a "streak marker" (the constant/shifted value described above). Since every row in an unbroken run of months shares the same marker, and a new streak always produces a different one, we can now identify which rows belong together.

# To isolate specifically the first streak, we look up the streak marker of each employee's very first (earliest-dated) record, and compare every row's own marker against that value. Rows that match belong to the first streak; rows that don't match belong to a later streak (i.e., after a gap occurred) and are discarded.

# Finally, within the rows that survive this filter (the first streak only), we pick the one with the maximum date — this is the last record of that first unbroken run — giving us exactly one row per employee: the commissionamt and monthlastdate at the point the first streak ended.












